# 04 — Evaluación del Modelo
**VeneCapital | Métricas, Curvas ROC y Matriz de Confusión**

Este notebook:
- Carga el mejor modelo guardado
- Evalúa sobre el conjunto de prueba (15%)
- Genera todas las visualizaciones
- Exporta el reporte de métricas en JSON

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from src.config import BEST_MODEL_PATH, REPORTS_DIR
from src.data_loader import get_datasets
from src.evaluate import full_evaluation, get_predictions, compute_metrics

print('✅ Imports exitosos')

In [ ]:
# ── Cargar modelo ─────────────────────────────────────────────────────────
print(f'Cargando modelo desde: {BEST_MODEL_PATH}')
model = tf.keras.models.load_model(str(BEST_MODEL_PATH))
print('✅ Modelo cargado')

In [ ]:
# ── Cargar dataset de prueba ──────────────────────────────────────────────
_, _, test_ds, _ = get_datasets()
print('✅ Dataset de prueba cargado')

In [ ]:
# ── Cargar historial de entrenamiento ─────────────────────────────────────
hist_path = REPORTS_DIR / 'training_history.json'
if hist_path.exists():
    with open(hist_path) as f:
        history_data = json.load(f)
    print('✅ Historial de entrenamiento cargado')
else:
    history_data = {}
    print('⚠️  No se encontró historial. Ejecuta primero el notebook 03.')

In [ ]:
# ── Evaluación completa ───────────────────────────────────────────────────
metrics = full_evaluation(model, test_ds, history_data)
print('\n📊 Métricas finales:')
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
# ── Mostrar visualizaciones generadas ─────────────────────────────────────
from IPython.display import Image, display
from src.config import CONFUSION_MATRIX_PATH, LEARNING_CURVES_PATH, ROC_CURVE_PATH

print('── Matriz de Confusión ──')
display(Image(filename=str(CONFUSION_MATRIX_PATH)))

print('── Curvas de Aprendizaje ──')
display(Image(filename=str(LEARNING_CURVES_PATH)))

print('── Curva ROC ──')
display(Image(filename=str(ROC_CURVE_PATH)))

In [ ]:
# ── Predicción sobre una imagen individual ────────────────────────────────
import cv2
from src.config import IMG_SIZE

def predict_single_image(image_path: str, model) -> dict:
    """Clasifica una imagen y retorna la predicción con confianza."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)  # Añadir dimensión de batch

    prob = float(model.predict(img, verbose=0)[0][0])
    clase = 'DEEPFAKE' if prob >= 0.5 else 'REAL'
    confianza = prob if prob >= 0.5 else 1 - prob

    return {'clase': clase, 'probabilidad_fake': prob, 'confianza': confianza}

# Ejemplo de uso:
# resultado = predict_single_image('../data/raw/real/alguna_imagen.jpg', model)
# print(resultado)

print('✅ Función predict_single_image disponible')
print('   Uso: predict_single_image("ruta/imagen.jpg", model)')